# Reproducing Jacobsen et al. (2022) from dandiset 000230

**Paper.** Jacobsen RI, Nair RR, Obenhaus HA, Donato F, Slettmoen T, Moser MB, Moser EI (2022).
All-viral tracing of monosynaptic inputs to single birthdate-defined neurons in the intact brain.
*Cell Reports Methods* 2, 100221.
[doi:10.1016/j.crmeth.2022.100221](https://doi.org/10.1016/j.crmeth.2022.100221). The claims file
(`claims/jacobsen-2022.yaml`) was written from the version of record and its supplement.

**Dandiset.** [DANDI:000230](https://dandiarchive.org/dandiset/000230/0.220506.1516), version
0.220506.1516 ([doi:10.48324/dandi.000230/0.220506.1516](https://doi.org/10.48324/dandi.000230/0.220506.1516)).
The paper's Data and code availability section cites this version, and it is also the most recent
published one, so it is pinned. Earlier versions (0.220422.0713, 0.220426.0814, 0.220426.1130) predate
the paper's citation and are not used; the draft has the same 9 assets and total size.

**What the dandiset holds.** Nine NWB files of the optrode recordings in the MEC: three sessions of
mouse 58313 (days 17, 18 and 19 after the rabies injection) and six of mouse 70375 (days 14 to 16, two
per day). Each holds an open field part (position and head direction), laser trials, and only the
curated units the figures show (12 units in all; the files say "Only a curated subset of tasks
and/or units are exported"). The histology of the single-cell tracing (Figures 2 to 4, S1, S2,
Tables S1 and S2) and the two-photon imaging (Figures 6 and 7) are not in it, and the paper names no
other deposit for them.

**Outside inputs.** None as data. Methods details for the cell classes come from Rowland et al.
(2018), which the STAR Methods cite for the analysis, and from the analysis code that paper links
(see the docstring of `mec_units.py`).

| Claim | Status |
|---|---|
| fig5c-functional-cell-types | tested |
| fig5c-responsive-grid-cell | tested |
| fig5c-example-cells | tested |
| figs3-stable-input-cell | tested |
| figs4-nonresponsive-cells | tested |
| responsive-cells-at-19-days | tested |
| mec-inputs-from-two-weeks | tested |
| fig2a-four-or-fewer-starters, fig2a-starter-count-distribution, fig2a-single-starter-rate, no-starter-cells-in-mec, fig2b-rabies-cells-increase-with-time, fig2b-conclusive-more-than-control, figs2a-spread-increases-with-time, figs2a-conclusive-spread-more-than-control, figs2b-mec-inputs-increase-with-time, controls-no-mec-spread, figs1a-no-expression-without-tva, figs1b-sparse-cre, fig3-inputs-reflect-starter-identity, fig4a-distant-granule-input, fig4b-granule-inputs-dominate, fig4-subgranular-inputs, fig4c-dg-starter-local-and-mec-inputs, cytotoxicity-after-30-days, population-targeting-mec-labelling | blocked, data missing (histology not deposited) |
| fig6b-imaged-input-cells, fig7b-dual-labelled-input-cell | blocked, data missing (imaging not deposited) |

In [ ]:
import hashlib
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import mec_units as mu

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 60)

started = time.time()
sessions = mu.load_all()
print(f"loaded {len(sessions)} files from dandiset {mu.DANDISET_ID} version {mu.DANDISET_VERSION} "
      f"in {time.time() - started:.0f} s")
units = []
for s in sessions:
    of = mu.open_field(s)
    s["of"] = of
    for u in s["units"]:
        units.append({"subject": s["subject"], "day": s["day"], "session": s["session_no"],
                      "label": f"T{u['tetrode']} C{u['cluster']}", "unit": u, "s": s})
print(len(units), "units")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
rows = []
for s in sessions:
    dur = s["ttl_off"] - s["ttl_on"]
    on = mu.pulses(s, "onset")
    isi = np.diff(on)
    rows.append({
        "file": s["path"].split("/")[1][:28], "subject": s["subject"], "sex": s["sex"],
        "start": s["session_start"][:16], "day": s["day"], "session": s["session_no"],
        "units": ",".join(str(u["unit_id"]) for u in s["units"]),
        "spikes": ",".join(str(len(u["spikes"])) for u in s["units"]),
        "quality": ",".join(sorted({u["quality"] for u in s["units"]})),
        "epochs": ";".join(f"{r.task}/{r.arena} {r.start:.0f}-{r.stop:.0f}s" for r in s["epochs"].itertuples()),
        "epoch_task_desc": ",".join(s["epochs"].task_desc),
        "ttl_events": len(s["ttl_on"]), "events_over_5ms": int((dur >= mu.MAX_PULSE_S).sum()),
        "pulses": len(on), "pulse_ms_median": round(1000 * np.median(dur[dur < mu.MAX_PULSE_S]), 3),
        "pulse_ms_range": f"{1000 * dur[dur < mu.MAX_PULSE_S].min():.3f}-{1000 * dur[dur < mu.MAX_PULSE_S].max():.3f}",
        "isi_s_values": ",".join(f"{v:.3f}" for v in np.unique(np.round(isi, 2))),
        "location": s["electrode_location"],
    })
info = pd.DataFrame(rows)
print("Files per subject:", info.subject.value_counts().to_dict())
print("Units per subject:", pd.Series([u["subject"] for u in units]).value_counts().to_dict())
print(info.drop(columns=["file"]).to_string(index=False))
print("\nSubject descriptions:")
for subj in sorted({s["subject"] for s in sessions}):
    print(" ", subj, "|", [s for s in sessions if s["subject"] == subj][0]["subject_description"].replace("\n", " "))
# Days against dates: the day difference between sessions of one mouse must equal the date difference.
for subj in sorted({s["subject"] for s in sessions}):
    ss = sorted([s for s in sessions if s["subject"] == subj], key=lambda s: s["session_start"])
    d0 = pd.Timestamp(ss[0]["session_start"][:10])
    print(f"{subj}: day - date offset per session:",
          [s["day"] - (pd.Timestamp(s["session_start"][:10]) - d0).days for s in ss])

The session descriptions give the day after the rabies injection and the session of the day, and
the day numbers agree with the session dates (the offset between day and date is constant within
each mouse). Mouse 58313 has sessions on days 17, 18 and 19, and 70375 two on each of days 14, 15
and 16, as Figures 5C, S3 and S4 state. All 12 units are labelled `Good`; unit ids read as tetrode
times 1000 plus cluster id (1089 is T1 C89), which matches the cells the figures name (T1 C89, T1 C57,
T1 C52, T4 C16, T4 C50, T1 C50). The units of 70375 carry a different cluster id in each session.

The open field is not an epoch: the epochs table holds only `PhotoStim` epochs in the holding box.
The dandiset description says sessions are labelled with the laser set point (20 mW set is 7.5 mW
actual, 40 mW is 15.1 mW). The eight single-epoch files carry it in `task_desc` (all "10Hz20mW", so
7.5 mW, as the legends of Figures S3 and S4 state); the two laser epochs of the day-17 file of 58313
carry "N/A", so their power is not known. The Methods describe "1 ms light pulses at 10 Hz continuously";
the TTL events show 1.0 to 1.1 ms pulses in trains of 6 at 10 Hz, a train every 2.6 s (intervals
of 0.1 and 2.1 s). The first TTL event of every epoch begins exactly at the epoch start and lasts
0.6 to 1.9 s; it is left out, so that 275 pulses remain in a 120 s epoch (276 events less one).

In [ ]:
rows = []
for s in sessions:
    of = s["of"]
    pos = s["pos"]
    spk = np.concatenate([u["spikes"] for u in s["units"]])
    calc_speed = of["speed"]
    raw_speed = np.hstack([0, np.hypot(*np.diff(pos, axis=0).T) * s["pos_rate"]])
    rows.append({
        "subject": s["subject"], "day": s["day"], "session": s["session_no"],
        "pos_samples": len(pos), "pos_rate_hz": s["pos_rate"], "open_field_s": of["duration"],
        "x_cm": f"{pos[:, 0].min():.1f} to {pos[:, 0].max():.1f}", "y_cm": f"{pos[:, 1].min():.1f} to {pos[:, 1].max():.1f}",
        "extent_cm": f"{np.ptp(pos[:, 0]):.0f} x {np.ptp(pos[:, 1]):.0f}",
        "nan_pos": int(np.isnan(pos).sum()),
        "hd_rad": f"{s['hd'].min():.3f} to {s['hd'].max():.3f}",
        "stored_speed_median": round(float(np.median(s["stored_speed"])), 2),
        "raw_speed_median": round(float(np.median(raw_speed)), 2),
        "smoothed_speed_median": round(float(np.median(calc_speed)), 2),
        "frac_time_over_5cm_s": round(float(np.mean(calc_speed > mu.SPEED_CUTOFF_CM_S)), 2),
        "spikes_s": f"{spk.min():.1f} to {spk.max():.1f}",
        "spikes_between_of_and_laser": int(((spk >= of["t_end"]) & (spk < s["epochs"].start.min())).sum()),
        "first_laser_epoch_s": s["epochs"].start.min(),
        "lfp_samples": s["lfp"]["n"], "lfp_t_last": s["lfp"]["t_last"], "lfp_backward_steps": s["lfp"]["n_backward_steps"],
    })
chk = pd.DataFrame(rows)
print(chk.to_string(index=False))
print("\nStated units: position 'meters' with conversion 0.01 (values in cm), head direction radians, speed m/s with conversion 0.01 (cm/s).")
# Recordings deposited more than once: compare spike trains and position series between files.
hashes = {}
for s in sessions:
    for name, arr in [("position", s["pos"])] + [(f"unit {u['unit_id']}", u["spikes"]) for u in s["units"]]:
        h = hashlib.sha1(np.ascontiguousarray(arr).tobytes()).hexdigest()
        hashes.setdefault(h, []).append(f"{s['subject']} day {s['day']} s{s['session_no']} {name}")
dups = [v for v in hashes.values() if len(v) > 1]
print("Identical arrays across files:", dups if dups else "none")
lens = pd.Series({f"{s['subject']} d{s['day']} s{s['session_no']}": len(s["pos"]) for s in sessions})
print("Position lengths shared by several files:", lens[lens.duplicated(keep=False)].to_dict())

Positions are in cm (values within about -5 to 158 cm) and span about 94 x 97 cm in eight files,
consistent with the 1 x 1 m open field of the Methods. In the day-17 file of 58313 they span only
about 76 x 75 cm; the file does not say why (a smaller box, or an animal that did not reach the
far side). Head direction lies in 0 to 2 pi radians. The position series are sampled at 50 Hz, as
the Methods state, and last 1800 to 1801 s (30 min, "approximately 30 min" in the Methods) except
day 14 session 2 of 70375 (2722 s). The open field is taken as the span of the position series; in eight of nine files there are
spikes after it and before the first laser epoch, so the recording continued between the two parts.
The stored speed series is in cm/s and its median is close to that of the speed computed from the
positions. No array is repeated between files, so no recording was deposited twice.

The LFP timestamps run backwards 3 or 4 times within each file and end at 120 to 180 s although
each LFP series has 510,250 to 815,750 samples 4 ms apart (over 2000 s); the LFP is not used here.

In [ ]:
# Clock check: spikes and laser TTLs. If the TTL times share the spike clock, spikes of a directly
# activated cell cluster at a fixed delay after the true pulses and not after pulse times shifted
# into the gap between pulses (50 ms later). Spikes and position: shown in the claim sections, where
# spatial scores are compared with time-shifted spike trains.
rows = []
for x in units:
    s, u = x["s"], x["unit"]
    p = mu.pulses(s, "onset")
    lf = mu.latency_fidelity(u["spikes"], p)
    lf_shift = mu.latency_fidelity(u["spikes"], p + 0.050)
    rows.append({"subject": x["subject"], "day": x["day"], "session": x["session"], "unit": x["label"],
                 "fidelity_true_pct": round(lf["fidelity_pct"], 2), "fidelity_shifted_50ms_pct": round(lf_shift["fidelity_pct"], 2),
                 "latency_true_ms": round(lf["latency_ms"], 2), "latency_shifted_ms": round(lf_shift["latency_ms"], 2)})
clock = pd.DataFrame(rows)
print(clock.to_string(index=False))

For the units that respond, fidelity after the true pulses (12.55 to 81.82%) far exceeds fidelity
after the same times shifted 50 ms (0.00 to 6.91%), and their latencies after the true pulses are
2.36 to 4.24 ms. This is consistent with the TTL events sharing the spike clock. It does not by itself show
the open-field tracking shares that clock; the spatial scores below, which exceed those of
time-shifted spike trains for the grid, head direction and border cells, are consistent with it.

## fig5c-functional-cell-types

> In mice with widespread RABV-ChR2-YFP expression, MEC recordings include grid, head direction and
> border cells, so the network as a whole is not disrupted by the rabies expression.

Criterion: grid, head direction and border cells classified with the scores and thresholds of
Rowland et al. (2018); reproduced when at least one unit of each type is found. Scores and
parameters are those of `mec_units.py` (docstring). The shuffle interval is run under both readings
of Rowland et al.'s "random interval between +/- 20 s" (`literal`, within 20 s; `standard`, at
least 20 s). Each unit has its own seeded generator.

In [ ]:
t0 = time.time()
cls_rows = []
for k, x in enumerate(units):
    c = mu.classify(x["unit"]["spikes"], x["s"]["of"], rng_seed=1000 + 10 * k)
    cls_rows.append({"subject": x["subject"], "day": x["day"], "session": x["session"], "unit": x["label"],
                     "of_spikes": len(mu.spike_samples(x["unit"]["spikes"], x["s"]["of"])), **c})
cls = pd.DataFrame(cls_rows)
print(f"classified {len(cls)} units in {time.time() - t0:.0f} s, {mu.N_SHUFFLES} shuffles per unit and reading")

In [ ]:
cols = ["subject", "day", "session", "unit", "of_spikes", "gridness", "thr_grid_literal", "thr_grid_standard",
        "mvl", "hd_stability", "thr_mvl_literal", "thr_mvl_standard", "border", "thr_border_literal", "thr_border_standard"]
print(cls[cols].round(3).to_string(index=False))
print()
print(cls[["subject", "day", "session", "unit", "grid_literal", "hd_literal", "border_literal",
           "grid_standard", "hd_standard", "border_standard"]].to_string(index=False))
for reading in ("literal", "standard"):
    n = {t: int(cls[f"{t}_{reading}"].sum()) for t in ("grid", "hd", "border")}
    print(f"\n{reading} shuffle: grid {n['grid']}, head direction {n['hd']}, border {n['border']} of {len(cls)} units;",
          "by subject:", cls.groupby("subject")[[f"grid_{reading}", f"hd_{reading}", f"border_{reading}"]].sum().to_dict("index"))
named = {"T1 C89": "grid (Fig 5C, day 19)", "T1 C50": "grid (Fig S4, day 19)", "T4 C16": "head direction (Fig S4, day 17)",
         "T4 C50": "border (Fig S4, day 18)"}
print("\nUnits the figures name:")
for lab, what in named.items():
    r = cls[(cls.subject == "58313") & (cls.unit == lab)].iloc[0]
    print(f"  {lab} {what}: gridness {r.gridness:.3f}, MVL {r.mvl:.3f} (stability {r.hd_stability:.2f}), border {r.border:.3f};"
          f" classes literal {[t for t in ('grid', 'hd', 'border') if r[f'{t}_literal']]}, standard {[t for t in ('grid', 'hd', 'border') if r[f'{t}_standard']]}")

In [ ]:
# Rate maps and head-direction tuning of the four named units (ranges printed below the figure).
fig, axes = plt.subplots(1, 5, figsize=(17, 3.2))
ranges = []
for ax, lab in zip(axes[:3], ["T1 C89", "T1 C50", "T4 C50"]):
    x = [x for x in units if x["subject"] == "58313" and x["label"] == lab][0]
    rm = mu.rate_map(mu.spike_samples(x["unit"]["spikes"], x["s"]["of"]), x["s"]["of"])
    ax.imshow(rm.T, origin="lower", cmap="jet")
    ax.set_title(f"{lab}, day {x['day']}, rate map")
    ax.axis("off")
    ranges.append(f"{lab} rate map 0 to {np.nanmax(rm):.2f} Hz")
x = [x for x in units if x["subject"] == "58313" and x["label"] == "T4 C16"][0]
c, tc = mu.hd_tuning(mu.spike_samples(x["unit"]["spikes"], x["s"]["of"]), x["s"]["of"])
axes[3].remove()
axp = fig.add_subplot(1, 5, 4, projection="polar")
axp.plot(np.append(c, c[0]), np.append(tc, tc[0]))
axp.set_title("T4 C16, day 17, HD tuning")
ranges.append(f"T4 C16 tuning {tc.min():.2f} to {tc.max():.2f} Hz")
ac = mu.autocorrelogram(mu.rate_map(mu.spike_samples(units[5]["unit"]["spikes"], units[5]["s"]["of"]), units[5]["s"]["of"]))
axes[4].imshow(ac.T, origin="lower", cmap="jet", vmin=-1, vmax=1)
axes[4].set_title(f"{units[5]['label']} autocorrelogram")
axes[4].axis("off")
plt.tight_layout()
plt.show()
print("; ".join(ranges), f"; autocorrelogram {np.nanmin(ac):.2f} to {np.nanmax(ac):.2f} (axis -1 to 1)")

**Result: reproduced, resting on an assumption.** Under the standard shuffle (shifts of at least
20 s) the 12 units include 2 grid cells (T1 C89 and T1 C50, gridness 0.854 and 0.996), 2 head
direction cells (T4 C16, mean vector length 0.912 and stability 0.97; and the 70375 unit of day 15
session 2, a weak one at 0.123 and 0.675) and 1 border cell (T4 C50, border score 0.656 against a
threshold of 0.555). Under the literal reading of the shuffle (shifts within 20 s) T4 C50 falls just
short (0.656 against 0.663), giving 2 grid, 1 head direction and 0 border cells, so the outcome rests
on adopting the standard shuffle, which the methods Rowland et al. (2018) cite use. Under it, each of
the four units the figures name is classed as the paper names it. All classified cells are from
58313 except the weak head direction cell. The dandiset holds only the 12 curated units shown in the
figures, so the part of the statement about the network as a whole cannot be tested beyond these
examples.

## fig5c-responsive-grid-cell

> Among the cells that responded to laser stimulation with a 2 to 4 ms latency, a sign of direct
> ChR2 activation and so of a monosynaptic projection to the hippocampus, were grid cells.

Criterion: latency and fidelity as defined in the claims file (STAR Methods), under both pulse-time
readings (a, TTL onset; b, TTL offset); responsive (R-a) when the latency lies in the widened range
1.8 to 4.4 ms and the fidelity is at least 11.64%. Reproduced when, under one reading, a unit classed
as a grid cell in fig5c-functional-cell-types is responsive. For the day-17 file, with two laser
epochs of unknown power, each epoch and both together are shown. The pre-pulse fidelity (a spike in
the 10 ms before the pulse) is a variant that does not decide.

In [ ]:
def laser_table(units_sel, lat_range):
    rows = []
    for x in units_sel:
        s, u = x["s"], x["unit"]
        epochs = [None] if len(s["epochs"]) == 1 else [None, 0, 1]
        for reading in ("onset", "offset"):
            for ep in epochs:
                lf = mu.latency_fidelity(u["spikes"], mu.pulses(s, reading, ep))
                rows.append({"subject": x["subject"], "day": x["day"], "session": x["session"], "unit": x["label"],
                             "pulse_time": reading, "epoch": "all" if ep is None else ep + 1,
                             "pulses": lf["n_pulses"], "with_spike": lf["n_hit"],
                             "latency_ms": round(lf["latency_ms"], 2), "fidelity_pct": round(lf["fidelity_pct"], 2),
                             "pre_pulse_pct": round(lf["pre_fidelity_pct"], 2),
                             "responsive": mu.responsive(lf, lat_range)})
    return pd.DataFrame(rows)

lt = laser_table(units, (1.8, 4.4))
lt = lt.merge(cls[["subject", "day", "session", "unit", "grid_literal", "grid_standard", "hd_literal", "hd_standard",
                   "border_literal", "border_standard"]], on=["subject", "day", "session", "unit"])
print(lt.to_string(index=False))
for reading in ("onset", "offset"):
    r = lt[lt.pulse_time == reading]
    resp_units = r[r.responsive].drop_duplicates(["subject", "day", "session", "unit"])
    print(f"\nreading {reading}: {len(resp_units)} unit-sessions responsive (1.8-4.4 ms, fidelity >= 11.64%) in some epoch;",
          "of them grid (literal shuffle):", resp_units[resp_units.grid_literal].unit.tolist(),
          "grid (standard shuffle):", resp_units[resp_units.grid_standard].unit.tolist())

**Result: reproduced.** Under reading (a), TTL onset, 9 unit-sessions are responsive, among them
T1 C89 (day 19), a grid cell under both shuffles, with latency 2.75 ms, fidelity 40.73% and no spike
in the 10 ms before any pulse (pre-pulse 0.00%). Under reading (b), TTL offset, latencies are about
1.06 ms shorter (the pulse width) and T1 C89, at 1.70 ms, is not responsive; 3 unit-sessions are,
none of them a grid cell.

## fig5c-example-cells

> Three example responsive cells of mouse 58313 had these open-field rates, latencies and
> fidelities; a grid cell (T1 C89, 19 days) 1.10 Hz, 2.75 ms, 40.73%; a high-rate cell (T1 C57,
> 17 days) 12.67 Hz, 3.90 ms, 35.64%; a low-rate cell (T1 C52, 17 days) 0.30 Hz, 2.65 ms, 11.64%.

The files keep the tetrode and cluster labels, so the units are found by label; each day of 58313
has one session in the dandiset. Both pulse-time readings, and for day 17 each laser epoch and both.

In [ ]:
expected = {"T1 C89": (19, 1.10, 2.75, 40.73), "T1 C57": (17, 12.67, 3.90, 35.64), "T1 C52": (17, 0.30, 2.65, 11.64)}
rows = []
for lab, (day, e_rate, e_lat, e_fid) in expected.items():
    x = [x for x in units if x["subject"] == "58313" and x["label"] == lab and x["day"] == day][0]
    rate = mu.open_field_rate(x["unit"]["spikes"], x["s"]["of"])
    epochs = [None] if len(x["s"]["epochs"]) == 1 else [None, 0, 1]
    for reading in ("onset", "offset"):
        for ep in epochs:
            lf = mu.latency_fidelity(x["unit"]["spikes"], mu.pulses(x["s"], reading, ep))
            rows.append({"unit": lab, "day": day, "session": x["session"], "pulse_time": reading,
                         "epoch": "all" if ep is None else ep + 1, "pulses": lf["n_pulses"],
                         "rate_hz": round(rate, 3), "exp_rate": e_rate, "rate_ok": abs(rate - e_rate) <= 0.1 * e_rate + 1e-9,
                         "latency_ms": round(lf["latency_ms"], 3), "exp_lat": e_lat, "lat_ok": abs(lf["latency_ms"] - e_lat) <= 0.1 * e_lat + 1e-9,
                         "fidelity_pct": round(lf["fidelity_pct"], 3), "exp_fid": e_fid,
                         "fid_ok": abs(lf["fidelity_pct"] - e_fid) <= 0.1 * e_fid + 1e-9})
ex = pd.DataFrame(rows)
ex["all_ok"] = ex.rate_ok & ex.lat_ok & ex.fid_ok
print(ex.to_string(index=False))
best = ex[(ex.pulse_time == "onset")]
print("\nReading (a), onset: rows within tolerance on all three values:",
      best[best.all_ok][["unit", "epoch"]].values.tolist())
print("Reading (b), offset: rows within tolerance on all three values:",
      ex[(ex.pulse_time == "offset") & ex.all_ok][["unit", "epoch"]].values.tolist())

**Result: reproduced, resting on an assumption.** Under reading (a) all nine values are within
tolerance when T1 C57 is taken from the second laser epoch of day 17 (12.670 Hz, 3.895 ms, 35.636%)
and T1 C52 from the first (0.303 Hz, 2.646 ms, 11.636%); T1 C89, whose session has one epoch, gives
1.098 Hz, 2.752 ms and 40.727%. With the two day-17 epochs pooled, T1 C57 still matches (4.237 ms,
32.545%) but T1 C52's latency (2.358 ms) falls 0.03 ms below its tolerance. No single epoch fits both
day-17 cells; the two epochs carry no power label and the paper does not say which laser epoch
Figure 5C shows, so taking the epoch per cell is an assumption. Under reading (b) no row has all
three values within tolerance.

## figs3-stable-input-cell

> The firing properties of one input cell stayed stable across six sessions on three days (14, 15
> and 16 days after the rabies injection); it responded to every laser session with a latency of
> 2.50 to 2.95 ms.

Criterion: mouse 70375; the unit sorted under the same cluster label in all six sessions when the
dandiset keeps one, otherwise the unit whose waveform best matches across sessions (an assumption).
Reproduced when the unit is found in all six sessions, every latency lies within 1.8 to 4.4 ms and all
18 values are within tolerance; partially reproduced when the unit is found and responsive (R-a) in
every session but some values miss. Each file of 70375 holds a single unit, all on tetrode 3 but
with six different cluster ids, so there is no shared label; the waveforms are compared below.

In [ ]:
s70 = sorted([x for x in units if x["subject"] == "70375"], key=lambda x: (x["day"], x["session"]))
print("Units of 70375 per session:", [(x["day"], x["session"], x["label"], x["unit"]["electrodes"]) for x in s70])
W = np.array([x["unit"]["waveform"] for x in s70])
print("waveform_mean shape per unit:", W.shape[1:], "(one channel per unit: the file stores the mean on one electrode)")
corr70 = np.corrcoef(W)
print("Waveform correlations between the six sessions:\n", pd.DataFrame(corr70, index=[f"d{x['day']}s{x['session']}" for x in s70],
      columns=[f"d{x['day']}s{x['session']}" for x in s70]).round(3).to_string())
others = [x for x in units if x["subject"] == "58313"]
cross = np.corrcoef(np.vstack([W, [x["unit"]["waveform"] for x in others]]))[:6, 6:]
print(f"\n70375 within-cell correlations: {corr70[np.triu_indices(6, 1)].min():.3f} to {corr70[np.triu_indices(6, 1)].max():.3f};"
      f" against the 6 units of 58313 (reference): {cross.min():.3f} to {cross.max():.3f}")
print("Peak-to-trough of waveform_mean per session, in the stored values (the file states volts):", [round(float(np.ptp(w)), 2) for w in W])

In [ ]:
exp_rate = [0.89, 1.06, 0.28, 1.61, 0.95, 1.84]
exp_lat = [2.83, 2.95, 2.77, 2.77, 2.58, 2.50]
exp_fid = [27.64, 48.00, 68.71, 64.36, 76.73, 81.82]
rows = []
for k, x in enumerate(s70):
    rate = mu.open_field_rate(x["unit"]["spikes"], x["s"]["of"])
    for reading in ("onset", "offset"):
        lf = mu.latency_fidelity(x["unit"]["spikes"], mu.pulses(x["s"], reading))
        rows.append({"day": x["day"], "session": x["session"], "unit": x["label"], "pulse_time": reading, "pulses": lf["n_pulses"],
                     "rate_hz": round(rate, 3), "exp_rate": exp_rate[k], "rate_ok": abs(rate - exp_rate[k]) <= 0.1 * exp_rate[k] + 1e-9,
                     "latency_ms": round(lf["latency_ms"], 3), "exp_lat": exp_lat[k], "lat_ok": abs(lf["latency_ms"] - exp_lat[k]) <= 0.1 * exp_lat[k] + 1e-9,
                     "fidelity_pct": round(lf["fidelity_pct"], 2), "exp_fid": exp_fid[k], "fid_ok": abs(lf["fidelity_pct"] - exp_fid[k]) <= 0.1 * exp_fid[k] + 1e-9,
                     "pre_pulse_pct": round(lf["pre_fidelity_pct"], 2), "responsive_2_4": mu.responsive(lf, (2.0, 4.0))})
s3 = pd.DataFrame(rows)
print(s3.to_string(index=False))
for reading in ("onset", "offset"):
    r = s3[s3.pulse_time == reading]
    n_ok = int(r.rate_ok.sum() + r.lat_ok.sum() + r.fid_ok.sum())
    print(f"reading {reading}: {n_ok} of 18 values within tolerance; latencies {r.latency_ms.min():.2f} to {r.latency_ms.max():.2f} ms;"
          f" responsive (R-a, 2-4 ms) in {int(r.responsive_2_4.sum())} of 6 sessions; missing: "
          f"{[(d, s_, q) for d, s_, ok_r, ok_l, ok_f in zip(r.day, r.session, r.rate_ok, r.lat_ok, r.fid_ok) for q, ok in (('rate', ok_r), ('latency', ok_l), ('fidelity', ok_f)) if not ok]}")

# Variants for day 14 session 2 (the longest file): the first 275 pulses, as in the other sessions,
# and the rate over the first 1800 s of the open field.
x = s70[1]
p = mu.pulses(x["s"], "onset")
lf275 = mu.latency_fidelity(x["unit"]["spikes"], p[:275])
of = x["s"]["of"]
spk = x["unit"]["spikes"]
print(f"\nDay 14 session 2: laser epoch {x['s']['epochs'].start[0]:.0f}-{x['s']['epochs'].stop[0]:.0f} s, {len(p)} pulses; "
      f"first 275 pulses: latency {lf275['latency_ms']:.2f} ms, fidelity {lf275['fidelity_pct']:.2f}% ({lf275['n_hit']}/275)")
print(f"open field {of['duration']:.0f} s; rate over all of it {mu.open_field_rate(spk, of):.3f} Hz; over the first 1800 s "
      f"{np.sum(spk < 1800) / 1800:.3f} Hz; over the last 1800 s {np.sum((spk >= of['duration'] - 1800) & (spk < of['duration'])) / 1800:.3f} Hz")
print("Classes of the 70375 unit per session (literal / standard shuffle):",
      cls[cls.subject == "70375"][["day", "session", "grid_literal", "hd_literal", "border_literal", "grid_standard", "hd_standard", "border_standard"]].values.tolist())

**Result: partially reproduced, resting on an assumption.** Each 70375 file holds one unit on
tetrode 3, with a different cluster id in each session; their mean waveforms correlate 0.918 to
0.998 (against -0.413 to 0.965 with the units of 58313), and they are taken as the same cell, an
assumption. Under reading (a) the unit is responsive in all six sessions with latencies 2.50 to
3.05 ms, and 17 of 18 values are within tolerance: the open-field rate of day 14 session 2 is 0.899 Hz
against 1.06 +/- 0.106 Hz. That file holds 2722 s of open field and 413 pulses; over its first 1800 s
the rate is 1.064 Hz, and over its first 275 pulses latency and fidelity are 2.95 ms and 48.00%, the
paper's values, which suggests the paper analysed the first 30 min and the first 275 pulses. That
variant is not the claims file's reading and does not decide. Under reading (b) the latencies are
1.44 to 1.99 ms and 11 of 18 values are within tolerance.

## figs4-nonresponsive-cells

> Some MEC cells recorded in the same area did not respond to laser stimulation, among them a head
> direction cell (T4 C16, 17 days, head direction score 0.81, mean vector length 0.90, peak rate
> 9.02 Hz, open-field rate 1.05 Hz), a border cell (T4 C50, 18 days, 1.69 Hz) and a grid cell (T1 C50,
> 19 days, 0.78 Hz).

Criterion: mouse 58313; the head direction score and mean vector length as in Rowland et al.
(2018); reproduced when the three units are not responsive (R-a) and the values are within
tolerance. Rowland et al. (2018) define one quantity, "Mean vector length (head-direction score)";
the paper reports two different values. The mean vector length of the tuning map is computed as the
helper does (archdata code: 180 bins, sigma 2 bins, no speed filter), and other readings are shown:
unsmoothed, speed-filtered, 6-degree bins, and the vector length of the head directions at spikes.

In [ ]:
rows = []
for lab, day, e_rate in [("T4 C16", 17, 1.05), ("T4 C50", 18, 1.69), ("T1 C50", 19, 0.78)]:
    x = [x for x in units if x["subject"] == "58313" and x["label"] == lab and x["day"] == day][0]
    rate = mu.open_field_rate(x["unit"]["spikes"], x["s"]["of"])
    epochs = [None] if len(x["s"]["epochs"]) == 1 else [None, 0, 1]
    for reading in ("onset", "offset"):
        for ep in epochs:
            lf = mu.latency_fidelity(x["unit"]["spikes"], mu.pulses(x["s"], reading, ep))
            rows.append({"unit": lab, "day": day, "pulse_time": reading, "epoch": "all" if ep is None else ep + 1,
                         "pulses": lf["n_pulses"], "with_spike": lf["n_hit"], "latency_ms": round(lf["latency_ms"], 2),
                         "fidelity_pct": round(lf["fidelity_pct"], 2), "pre_pulse_pct": round(lf["pre_fidelity_pct"], 2),
                         "responsive_2_4": mu.responsive(lf, (2.0, 4.0)), "responsive_1.8_4.4": mu.responsive(lf, (1.8, 4.4)),
                         "rate_hz": round(rate, 3), "exp_rate": e_rate, "rate_ok": abs(rate - e_rate) <= 0.1 * e_rate + 1e-9})
s4 = pd.DataFrame(rows)
print(s4.to_string(index=False))
print("Any of the three responsive under any reading or epoch:", bool(s4["responsive_1.8_4.4"].any()))

x = [x for x in units if x["subject"] == "58313" and x["label"] == "T4 C16"][0]
of = x["s"]["of"]
smp = mu.spike_samples(x["unit"]["spikes"], of)
fast = of["speed"] > mu.SPEED_CUTOFF_CM_S
hd_rows = []
for name, sel, smooth, nb in [("archdata: 180 bins, sigma 2, all samples", None, True, 180),
                              ("180 bins, unsmoothed", None, False, 180),
                              ("180 bins, sigma 2, speed > 5 cm/s", fast, True, 180),
                              ("60 bins (6 deg), unsmoothed", None, False, 60)]:
    if nb == 180:
        c, tc = mu.hd_tuning(smp, of, sel=sel, smooth=smooth)
    else:
        e = np.linspace(0, 2 * np.pi, nb + 1)
        occ, _ = np.histogram(of["hd"], e)
        cnt, _ = np.histogram(of["hd"][smp], e)
        tc = cnt / (occ / mu.POS_RATE_HZ)
        c = e[:-1] + np.pi / nb
    hd_rows.append({"reading": name, "mvl": round(mu.mean_vector_length(c, tc), 3), "peak_hz": round(float(tc.max()), 2)})
hd_rows.append({"reading": "vector length of head directions at spikes", "mvl": round(float(np.abs(np.mean(np.exp(1j * of["hd"][smp])))), 3), "peak_hz": np.nan})
hd = pd.DataFrame(hd_rows)
hd["mvl_vs_0.90_ok"] = (hd.mvl - 0.90).abs() <= 0.09
hd["mvl_vs_0.81_ok"] = (hd.mvl - 0.81).abs() <= 0.081
hd["peak_vs_9.02_ok"] = (hd.peak_hz - 9.02).abs() <= 0.902
print("\nT4 C16 head direction measures (expected HD score 0.81 +/- 0.081, MVL 0.90 +/- 0.09, peak 9.02 +/- 0.90 Hz):")
print(hd.to_string(index=False))
print(f"T4 C16 tuning stability (first against second half): {cls[(cls.subject == '58313') & (cls.unit == 'T4 C16')].hd_stability.iloc[0]:.3f}")

**Result: partially reproduced, resting on an assumption.** None of the three units is responsive
under either reading or any epoch (fidelity 0.00 to 1.82%), and their open-field rates (1.047, 1.692
and 0.776 Hz) match. For T4 C16 the mean vector length of the tuning map is 0.912 (0.90 expected).
The paper's separate head direction score of 0.81 is matched only by the vector length of the head
directions at spikes (0.880), a measure Rowland et al. (2018) do not define, and the peak rate of the
tuning curve is 10.62 Hz with the archdata parameters and 10.77 to 13.92 Hz under the other readings,
against 9.02 +/- 0.90 Hz. Rowland et al. (2018) name one quantity, "Mean vector length (head-direction
score)", so the definition of the score the paper reports apart from the mean vector length, and the
tuning-curve parameters behind Figure S4, are open; two of the six values miss under the cited
method.

## responsive-cells-at-19-days

> Laser-responsive cells with clear spatial firing patterns could be identified for at least 19
> days after the rabies injection.

Criterion: reproduced when a session 19 or more days after the rabies injection holds a unit that is
responsive (R-a: latency 2 to 4 ms, fidelity at least 11.64%) and classed as a grid, head direction or
border cell under Rowland et al. (2018). The widened 1.8 to 4.4 ms window is shown as well.

In [ ]:
late = lt[lt.day >= 19].copy()
late["responsive_2_4"] = [mu.responsive({"latency_ms": a, "fidelity_pct": b}, (2.0, 4.0)) for a, b in zip(late.latency_ms, late.fidelity_pct)]
late["spatial_literal"] = late.grid_literal | late.hd_literal | late.border_literal
late["spatial_standard"] = late.grid_standard | late.hd_standard | late.border_standard
print(late[["subject", "day", "session", "unit", "pulse_time", "latency_ms", "fidelity_pct", "responsive_2_4", "responsive",
            "spatial_literal", "spatial_standard"]].rename(columns={"responsive": "responsive_1.8_4.4"}).to_string(index=False))
print("Sessions at 19 days or more:", sorted({(s["subject"], s["day"], s["session_no"]) for s in sessions if s["day"] >= 19}))
for reading in ("onset", "offset"):
    r = late[late.pulse_time == reading]
    print(f"reading {reading}: responsive (2-4 ms) and spatial (literal shuffle): {r[r.responsive_2_4 & r.spatial_literal].unit.tolist()};"
          f" (standard shuffle): {r[r.responsive_2_4 & r.spatial_standard].unit.tolist()}")

**Result: reproduced.** The one session at 19 days or more (58313, day 19) holds T1 C89, responsive
under reading (a) (2.75 ms, 40.73%) and classed as a grid cell under both shuffles. Under reading (b)
its latency is 1.70 ms and no unit of that session qualifies.

## mec-inputs-from-two-weeks

> In the optogenetic experiments, input cells could be detected in the MEC from about 2 weeks after
> the rabies injection.

Criterion: read as detection by recording; reproduced when the earliest session holding a
responsive unit (R-a) is 12.6 to 15.4 days after the rabies injection.

In [ ]:
lt["responsive_2_4"] = [mu.responsive({"latency_ms": a, "fidelity_pct": b}, (2.0, 4.0)) for a, b in zip(lt.latency_ms, lt.fidelity_pct)]
for reading in ("onset", "offset"):
    r = lt[(lt.pulse_time == reading) & lt.responsive_2_4]
    by_subj = r.groupby("subject").day.min().to_dict()
    print(f"reading {reading}: earliest day with a responsive unit (2-4 ms, >= 11.64%): {r.day.min()} (per mouse {by_subj});"
          f" units on that day: {r[r.day == r.day.min()][['subject', 'session', 'unit', 'latency_ms', 'fidelity_pct']].values.tolist()}")
print("Earliest session day in the dandiset, per mouse:", {subj: min(s["day"] for s in sessions if s["subject"] == subj) for subj in sorted({s["subject"] for s in sessions})})

**Result: reproduced.** Under reading (a) the earliest session with a responsive unit is on day 14
(70375, both sessions: 2.83 ms and 27.64%; 3.05 ms and 46.49%), within 12.6 to 15.4 days. Under
reading (b), with the 2 to 4 ms window, it is day 17. The dandiset holds no session before day 14,
so it cannot show that responsive cells were absent earlier; the result is consistent with the
statement.

## Summary

| Claim | Outcome |
|---|---|
| fig5c-functional-cell-types | reproduced (assumed: standard shuffle) |
| fig5c-responsive-grid-cell | reproduced |
| fig5c-example-cells | reproduced (assumed: laser epoch per cell on day 17) |
| figs3-stable-input-cell | partially reproduced (assumed: same cell by waveform); one rate of 18 values misses |
| figs4-nonresponsive-cells | partially reproduced (assumed: head direction score definition); HD score and peak rate miss |
| responsive-cells-at-19-days | reproduced |
| mec-inputs-from-two-weeks | reproduced |
| 19 histology claims (Figures 2 to 4, S1, S2, Tables S1, S2, cytotoxicity, population labelling) | blocked, data missing |
| fig6b-imaged-input-cells, fig7b-dual-labelled-input-cell | blocked, data missing |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

**How the analysis differs from the paper's.** The paper's own analysis code is not published. Cell
classes follow the Methods of Rowland et al. (2018), with the details the text leaves open taken from
that paper's linked code (archdata, commit 9452263); the shuffle interval was run under two readings.
The pulse time was run as TTL onset and offset; the onset reproduces the printed latencies to the
second decimal. Pulses exclude the first TTL event of each epoch. The open field is the span of the
position series, as the files have no open-field epoch.

**What was found about the dataset.** It holds only the optrode recordings, and of those only the 12
curated units the figures show; the histology and imaging behind 21 of the 28 claims are not in it
and the paper names no other deposit. The open field has no epoch. The two laser epochs of the day-17
file carry no power label. The first TTL event of every epoch is spurious (onset at the epoch start,
0.6 to 1.9 s long). The pulses come in trains of 6 at 10 Hz every 2.6 s, where the Methods say 10 Hz
continuously. The LFP timestamps run backwards within each file. `waveform_mean` is labelled volts
but holds values with peak-to-trough 31 to 41. For day 14 session 2 of 70375, the file holds more
open field and more pulses than the paper's printed values use.

In [ ]:
print(f"notebook ran in {time.time() - started:.0f} s")